# Convert Fine-Tuned Keras Model to Mobile Formats

This notebook converts **final_model.keras** into:
- Android → `.tflite`
- iOS → `.mlmodel`
- Labels → `dish_labels.txt`

✔ Safe Kaggle input path
✔ Unique output names (no overwrite)
✔ Food-101 compatible


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from pathlib import Path
import time

print("TensorFlow:", tf.__version__)

In [ ]:
# ================= CONFIG =================

MODEL_PATH = "/kaggle/input/final-model/keras/default/1/final_model.keras"

RUN_ID = time.strftime("%Y%m%d_%H%M%S")

TFLITE_OUT = f"vision_final_{RUN_ID}.tflite"
COREML_OUT = f"vision_final_{RUN_ID}.mlmodel"
LABELS_OUT = f"dish_labels_{RUN_ID}.txt"

print("📦 Model path:", MODEL_PATH)
assert Path(MODEL_PATH).exists(), "❌ final_model.keras NOT FOUND"

In [ ]:
# ================= LOAD MODEL =================

print("🔄 Loading model...")
model = tf.keras.models.load_model(MODEL_PATH)

num_classes = model.output_shape[-1]
print("✅ Model loaded")
print("📊 Classes:", num_classes)

In [ ]:
# ================= CLASS LABELS =================

print("🔍 Loading Food-101 labels...")
info = tfds.builder("food101").info
class_names = info.features["label"].names

assert len(class_names) == num_classes, "❌ Label count mismatch"

print("✅ Labels loaded:", len(class_names))
print("Sample:", class_names[:5])

In [ ]:
# ================= TFLITE CONVERSION =================

print("🔄 Converting to TFLite (Android)...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open(TFLITE_OUT, "wb") as f:
    f.write(tflite_model)

print(f"✅ Saved {TFLITE_OUT} ({len(tflite_model)/1024/1024:.2f} MB)")

In [ ]:
# ================= COREML CONVERSION =================

print("🔄 Installing CoreML tools...")
!pip install -q protobuf==3.20.3 coremltools

import coremltools as ct

print("🔄 Converting to CoreML (iOS)...")

coreml_model = ct.convert(
    model,
    inputs=[ct.ImageType(
        name="image",
        shape=(1, 224, 224, 3),
        scale=1/255.0
    )],
    classifier_config=ct.ClassifierConfig(class_names)
)

coreml_model.short_description = "Food-101 EfficientNet (Fine-Tuned)"
coreml_model.save(COREML_OUT)

print(f"✅ Saved {COREML_OUT}")

In [ ]:
# ================= LABEL FILE =================

with open(LABELS_OUT, "w") as f:
    for name in class_names:
        f.write(name + "\n")

print(f"✅ Saved {LABELS_OUT}")

In [ ]:
# ================= SUMMARY =================

print("\n" + "="*60)
print("🎉 MOBILE EXPORT COMPLETE")
print("="*60)
print("📦 Download from Output tab:")
print("•", TFLITE_OUT)
print("•", COREML_OUT)
print("•", LABELS_OUT)